# Transformer quantum state: a four-spin tutorial

This notebook takes one very small system all the way through the TQS/VMC calculation. Because four spins have only $2^4=16$ basis configurations, we can inspect every number and compare sampling with exact enumeration.

By the end, you will see:

1. how an autoregressive model produces a normalized wavefunction;
2. what the causal transformer receives and predicts;
3. how the Heisenberg Hamiltonian connects spin configurations;
4. how local energies give the variational energy; and
5. what changes during one VMC/Adam update.

We use an open four-site spin-$1/2$ Heisenberg chain,

$$
H=\sum_{i=1}^{3}\mathbf S_i\cdot\mathbf S_{i+1}.
$$


## 1. Imports and the small TQS

The model width and number of layers are deliberately small. `seed=7` makes initialization and the tutorial output reproducible.

In [1]:
from pathlib import Path
import sys

import numpy as np

# Make the notebook runnable from either the repository root or examples/.
cwd = Path.cwd().resolve()
repo_root = next(path for path in (cwd, *cwd.parents) if (path / "pyqed").is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from pyqed.ml import TQS, heisenberg_connections

n_spins = 4
state = TQS(
    n_spins,
    d_model=16,
    n_heads=4,
    n_layers=2,
    seed=7,
)

print(f"sites: {state.n_visible}")
print(f"transformer width: {state.d_model}")
print(f"attention heads: {state.n_heads}")
print(f"transformer layers: {state.n_layers}")
print("\nSelected parameter shapes:")
for name in ("token_embedding", "position_embedding", "wq", "probability_output", "phase_output"):
    print(f"  {name:20s} {tuple(state.parameters[name].shape)}")

sites: 4
transformer width: 16
attention heads: 4
transformer layers: 2

Selected parameter shapes:
  token_embedding      (3, 16)
  position_embedding   (4, 16)
  wq                   (2, 16, 16)
  probability_output   (16, 2)
  phase_output         (16, 2)


## 2. Autoregressive normalization

For a configuration $\mathbf s=(s_1,s_2,s_3,s_4)$, where each $s_i\in\{-1,+1\}$, the TQS factorizes its Born probability:

$$
P_\theta(\mathbf s)=|\psi_\theta(\mathbf s)|^2
=\prod_{i=1}^{4}P_\theta(s_i\mid s_1,\ldots,s_{i-1}).
$$

Every conditional distribution is produced by a two-class softmax and therefore sums to one. Consequently, the product defines an exactly normalized joint probability—no partition function is needed.

The wavefunction also needs a phase:

$$
\psi_\theta(\mathbf s)
=\sqrt{P_\theta(\mathbf s)}\,e^{i\phi_\theta(\mathbf s)}.
$$


In [2]:
configurations = state.all_configurations()
amplitudes = np.asarray(state.amplitude(configurations))
probabilities = np.abs(amplitudes) ** 2

print("Number of configurations:", len(configurations))
print("Sum of probabilities:    ", probabilities.sum())
print("Wavefunction norm:       ", np.vdot(amplitudes, amplitudes).real)

Number of configurations: 16
Sum of probabilities:     0.9999999999999997
Wavefunction norm:        0.9999999999999998


## 3. What the causal transformer sees

The token convention in `TQS` is:

- `0`: start token;
- `1`: spin $-1$;
- `2`: spin $+1$.

To evaluate $(s_1,s_2,s_3,s_4)$, the network receives `[START, s1, s2, s3]`. Its four output positions predict `[s1, s2, s3, s4]`. A lower-triangular causal mask prevents position $i$ from using spins to its right.

In [3]:
target = np.array([-1, +1, -1, +1])
spin_tokens = ((target + 1) // 2) + 1
input_tokens = np.concatenate(([0], spin_tokens[:-1]))

print("physical configuration:", target)
print("transformer input:     ", input_tokens, "= [START, s1, s2, s3]")
print("predicted spins:       ", target, "= [s1, s2, s3, s4]")
print("\nCausal attention mask (1 means visible):")
print(np.tril(np.ones((n_spins, n_spins), dtype=int)))

physical configuration: [-1  1 -1  1]
transformer input:      [0 1 2 1] = [START, s1, s2, s3]
predicted spins:        [-1  1 -1  1] = [s1, s2, s3, s4]

Causal attention mask (1 means visible):
[[1 0 0 0]
 [1 1 0 0]
 [1 1 1 0]
 [1 1 1 1]]


We can recover the four conditional factors from the exactly enumerated joint distribution. This is only a teaching check; the implementation obtains them directly from the transformer's softmax outputs.

In [4]:
def prefix_probability(prefix):
    prefix = np.asarray(prefix)
    if prefix.size == 0:
        return 1.0
    matches = np.all(configurations[:, : prefix.size] == prefix, axis=1)
    return probabilities[matches].sum()

conditional_factors = []
for site in range(n_spins):
    shorter = prefix_probability(target[:site])
    longer = prefix_probability(target[: site + 1])
    conditional = longer / shorter
    conditional_factors.append(conditional)
    print(f"P(s{site + 1}={target[site]:+d} | previous spins) = {conditional:.8f}")

target_row = np.flatnonzero(np.all(configurations == target, axis=1))[0]
print("\nProduct of conditionals:", np.prod(conditional_factors))
print("Joint probability:      ", probabilities[target_row])

P(s1=-1 | previous spins) = 0.53283918
P(s2=+1 | previous spins) = 0.43935328
P(s3=-1 | previous spins) = 0.54325363
P(s4=+1 | previous spins) = 0.45756165

Product of conditionals: 0.05819186429004473
Joint probability:       0.05819186429004473


## 4. Inspect the complete wavefunction

`amplitude` evaluates $\psi_\theta(\mathbf s)$, while `abs(amplitude)**2` gives the sampling probability. The phase column is important: the antiferromagnetic ground state requires a nontrivial sign/phase structure.

In [5]:
print(" idx       spins          probability       Re(psi)       Im(psi)    phase")
print("-" * 82)
for index, (spins, probability, amplitude) in enumerate(
    zip(configurations, probabilities, amplitudes)
):
    spin_text = " ".join(f"{spin:+d}" for spin in spins)
    print(
        f"{index:4d}  [{spin_text}]  {probability:14.9f}  "
        f"{amplitude.real:12.7f}  {amplitude.imag:12.7f}  {np.angle(amplitude):8.4f}"
    )

 idx       spins          probability       Re(psi)       Im(psi)    phase
----------------------------------------------------------------------------------
   0  [+1 +1 +1 +1]     0.046619765     0.2158558     0.0051015    0.0236
   1  [+1 +1 +1 -1]     0.050653544     0.1786376     0.1369020    0.6539
   2  [+1 +1 -1 +1]     0.053091211     0.1483022     0.1763453    0.8716
   3  [+1 +1 -1 -1]     0.062985286     0.1118963     0.2246431    1.1087
   4  [+1 -1 +1 +1]     0.053342614     0.2173694     0.0780586    0.3448
   5  [+1 -1 +1 -1]     0.057819135     0.1383743     0.1966512    0.9576
   6  [+1 -1 -1 +1]     0.065259584     0.1679868     0.1924578    0.8532
   7  [+1 -1 -1 -1]     0.077389682     0.1337988     0.2439007    1.0690
   8  [-1 +1 +1 +1]     0.051321054     0.2207986    -0.0506857   -0.2256
   9  [-1 +1 +1 -1]     0.055605391     0.2186111     0.0884002    0.3843
  10  [-1 +1 -1 +1]     0.058191864     0.1997266     0.1352818    0.5954
  11  [-1 +1 -1 -1]     0.06

## 5. Hamiltonian connectivity

For one bond, the Heisenberg interaction is

$$
\mathbf S_i\cdot\mathbf S_j
=S_i^zS_j^z+\frac12\left(S_i^+S_j^-+S_i^-S_j^+\right).
$$

The diagonal contribution is $s_i s_j/4$. If neighboring spins differ, the off-diagonal term flips both with matrix element $1/2$. Therefore each configuration has only $O(N)$ connected configurations; a scalable VMC calculation does not construct a $2^N\times2^N$ matrix.

In [6]:
target_connected, target_elements = heisenberg_connections(target[None, :])
target_connected = np.asarray(target_connected[0])
target_elements = np.asarray(target_elements[0])

print("Connections from", target)
print("The first row is the diagonal term; the remaining rows correspond to bonds.")
for connected_state, matrix_element in zip(target_connected, target_elements):
    print(f"  {connected_state}    H(s,s') = {matrix_element:+.3f}")

Connections from [-1  1 -1  1]
The first row is the diagonal term; the remaining rows correspond to bonds.
  [-1  1 -1  1]    H(s,s') = -0.750
  [ 1 -1 -1  1]    H(s,s') = +0.500
  [-1 -1  1  1]    H(s,s') = +0.500
  [-1  1  1 -1]    H(s,s') = +0.500


For four spins only, we assemble the dense Hamiltonian as an exact reference. This is a validation tool, not the scalable TQS path.

In [7]:
connected, matrix_elements = heisenberg_connections(configurations)
connected = np.asarray(connected)
matrix_elements = np.asarray(matrix_elements)

# Convert each +/-1 configuration to its row/column integer label.
binary_weights = 2 ** np.arange(n_spins - 1, -1, -1)
connected_labels = ((connected == -1) * binary_weights).sum(axis=2)
rows = np.broadcast_to(np.arange(2**n_spins)[:, None], connected_labels.shape)

hamiltonian = np.zeros((2**n_spins, 2**n_spins), dtype=complex)
np.add.at(
    hamiltonian,
    (rows.ravel(), connected_labels.ravel()),
    matrix_elements.ravel(),
)

exact_eigenvalues, exact_eigenvectors = np.linalg.eigh(hamiltonian)
print("Hermitian check:", np.allclose(hamiltonian, hamiltonian.conj().T))
print("Exact eigenvalues:", np.round(exact_eigenvalues, 8))
print("Exact ground-state energy:", exact_eigenvalues[0])

Hermitian check: True
Exact eigenvalues: [-1.6160254  -0.95710678 -0.95710678 -0.95710678 -0.25       -0.25
 -0.25        0.1160254   0.45710678  0.45710678  0.45710678  0.75
  0.75        0.75        0.75        0.75      ]
Exact ground-state energy: -1.6160254037844386


## 6. Local energy

For a sampled configuration $\mathbf s$, VMC uses

$$
E_{\mathrm{loc}}(\mathbf s)
=\sum_{\mathbf s'}H_{\mathbf s\mathbf s'}
\frac{\psi_\theta(\mathbf s')}{\psi_\theta(\mathbf s)}.
$$

Only connected $\mathbf s'$ are required. The variational energy is the probability-weighted average

$$
E_\theta=\sum_{\mathbf s}P_\theta(\mathbf s)E_{\mathrm{loc}}(\mathbf s).
$$

Individual local energies can be complex, but their exact expectation is real for a Hermitian Hamiltonian.

In [8]:
local_energies = np.asarray(
    state.local_energies(configurations, connected, matrix_elements)
)
energy_from_local = np.sum(probabilities * local_energies)
energy_from_matrix = np.vdot(amplitudes, hamiltonian @ amplitudes)

print(" idx       spins          P(s)         Re(E_loc)      Im(E_loc)")
print("-" * 70)
for index, (spins, probability, local_energy) in enumerate(
    zip(configurations, probabilities, local_energies)
):
    spin_text = " ".join(f"{spin:+d}" for spin in spins)
    print(
        f"{index:4d}  [{spin_text}]  {probability:11.7f}  "
        f"{local_energy.real:14.8f}  {local_energy.imag:14.8f}"
    )

print("\nProbability-weighted local energy:", energy_from_local)
print("Direct matrix expectation:         ", energy_from_matrix)
print("They agree:", np.allclose(energy_from_local, energy_from_matrix))

 idx       spins          P(s)         Re(E_loc)      Im(E_loc)
----------------------------------------------------------------------
   0  [+1 +1 +1 +1]    0.0466198      0.75000000      0.00000000
   1  [+1 +1 +1 -1]    0.0506535      0.74981074      0.11054547
   2  [+1 +1 -1 +1]    0.0530912      0.66009421     -0.35744947
   3  [+1 +1 -1 -1]    0.0629853      0.72360186     -0.07208253
   4  [+1 -1 +1 +1]    0.0533426      0.59397853     -0.01403139
   5  [+1 -1 +1 -1]    0.0578191      0.70614563     -0.24283598
   6  [+1 -1 -1 +1]    0.0652596      0.67461196     -0.07132902
   7  [+1 -1 -1 -1]    0.0773897      0.70604441     -0.12197242
   8  [-1 +1 +1 +1]    0.0513211      0.67904813      0.27525500
   9  [-1 +1 +1 -1]    0.0556054      0.67846682      0.38374024
  10  [-1 +1 -1 +1]    0.0581919      0.55883190     -0.37806390
  11  [-1 +1 -1 -1]    0.0689863      0.68878590     -0.11588513
  12  [-1 -1 +1 +1]    0.0629326      0.54490222      0.37973710
  13  [-1 -1 +1 -1] 

## 7. Direct autoregressive sampling

Sampling proceeds left-to-right. The TQS draws $s_1$, feeds it back to predict $s_2$, and continues. Samples are independent; no Metropolis chain, burn-in, or acceptance step is needed.

The empirical frequencies below approach the exact probabilities as the number of samples grows.

In [9]:
samples = state.sample(20_000, seed=123)
sample_labels = ((samples == -1) * binary_weights).sum(axis=1)
frequencies = np.bincount(sample_labels, minlength=2**n_spins) / len(samples)

print(" idx       spins            exact P    sampled frequency")
print("-" * 62)
for index, (spins, probability, frequency) in enumerate(
    zip(configurations, probabilities, frequencies)
):
    spin_text = " ".join(f"{spin:+d}" for spin in spins)
    print(f"{index:4d}  [{spin_text}]    {probability:10.6f}    {frequency:10.6f}")

print("\nLargest absolute frequency error:", np.max(np.abs(frequencies - probabilities)))

 idx       spins            exact P    sampled frequency
--------------------------------------------------------------
   0  [+1 +1 +1 +1]      0.046620      0.050100
   1  [+1 +1 +1 -1]      0.050654      0.050200
   2  [+1 +1 -1 +1]      0.053091      0.053300
   3  [+1 +1 -1 -1]      0.062985      0.060100
   4  [+1 -1 +1 +1]      0.053343      0.055250
   5  [+1 -1 +1 -1]      0.057819      0.058800
   6  [+1 -1 -1 +1]      0.065260      0.064800
   7  [+1 -1 -1 -1]      0.077390      0.077150
   8  [-1 +1 +1 +1]      0.051321      0.051500
   9  [-1 +1 +1 -1]      0.055605      0.056650
  10  [-1 +1 -1 +1]      0.058192      0.057000
  11  [-1 +1 -1 -1]      0.068986      0.068250
  12  [-1 -1 +1 +1]      0.062933      0.058350
  13  [-1 -1 +1 -1]      0.067981      0.068800
  14  [-1 -1 -1 +1]      0.076840      0.078350
  15  [-1 -1 -1 -1]      0.090981      0.091400

Largest absolute frequency error: 0.0045825522193007895


## 8. One VMC optimization update

`train_step` performs the complete scalable loop:

1. draw configurations from $P_\theta=|\psi_\theta|^2$;
2. generate only their Hamiltonian-connected configurations;
3. evaluate their local energies;
4. differentiate the centered VMC estimator; and
5. update every transformer parameter with Adam.

The reported `state.energy` is the Monte Carlo estimate from samples drawn **before** the parameter update. Since this system is tiny, we additionally enumerate all 16 states to measure the exact variational energy before and after the update.

In [10]:
def exact_variational_energy(model):
    psi = np.asarray(model.state_vector())
    return np.vdot(psi, hamiltonian @ psi).real

parameters_before = {
    name: np.asarray(value).copy() for name, value in state.parameters.items()
}
energy_before = exact_variational_energy(state)

state.train_step(
    heisenberg_connections,
    n_samples=4096,
    learning_rate=3.0e-3,
)

energy_after = exact_variational_energy(state)
parameter_change = np.sqrt(
    sum(
        np.sum((np.asarray(state.parameters[name]) - old_value) ** 2)
        for name, old_value in parameters_before.items()
    )
)

print(f"Exact variational energy before update: {energy_before: .10f}")
print(f"Monte Carlo estimate used by update:    {state.energy.real: .10f}")
print(f"Exact variational energy after update:  {energy_after: .10f}")
print(f"Exact ground-state energy:              {exact_eigenvalues[0]: .10f}")
print(f"VMC local-energy variance:              {state.energy_variance: .10f}")
print(f"Total parameter-vector change:          {parameter_change: .10f}")

Exact variational energy before update:  0.6834237556
Monte Carlo estimate used by update:     0.6823895948
Exact variational energy after update:   0.2492969297
Exact ground-state energy:              -1.6160254038
VMC local-energy variance:               0.0504959024
Total parameter-vector change:           0.2444709693


One stochastic update need not lower the energy every time, although it does for this fixed seed. Repeated updates gradually adjust both the probabilities and phases toward the ground state. The four-spin example in `examples/four_spin_tqs.py` runs 300 updates and compares against exact diagonalization.

Set the switch below to `True` if you want to watch a shorter training run. Exact enumeration is used only for the printed diagnostic.

In [11]:
RUN_MORE_STEPS = False

if RUN_MORE_STEPS:
    for step in range(1, 51):
        state.train_step(
            heisenberg_connections,
            n_samples=2048,
            learning_rate=3.0e-3,
        )
        if step == 1 or step % 10 == 0:
            energy = exact_variational_energy(state)
            print(f"step {step:2d} | TQS {energy: .10f} | exact {exact_eigenvalues[0]: .10f}")
else:
    print("Set RUN_MORE_STEPS = True to continue training.")

Set RUN_MORE_STEPS = True to continue training.


## 9. Map the notebook back to the source

- `pyqed/ml/tqs.py`: embeddings, causal self-attention, probability/phase heads, and cached direct sampling.
- `pyqed/ml/autoregressive.py`: the public amplitude API, local energies, VMC/Adam update, and Hamiltonian connectivity.
- `examples/four_spin_tqs.py`: the complete 300-step validation run.

The scalable path never enumerates all $2^N$ configurations and never builds the dense Hamiltonian. It stores $O(N)$ connections per sampled configuration and uses direct autoregressive samples. Enumeration in this notebook exists solely so that every step can be independently checked.